In [ ]:
plan = """
iterate through (models x subsets) configurations
[ ] aggregate results in different weight decomposition strategies
    1. whole layer
    2. mlp in layer
    3. specific weights in mlp: gate, up, down
    4. attn in layer: q, k, v, o
    5. aggregate results across all strategies above, rank the best
    -> each save into a  ablation/{model}/strategies/{subset}_{name}.tsv
[ ] select the best strategies (k=3) from each of categories above and
    plot score distributions of decisive error steps vs. non- counterparts.
    -> save as one plot at ablation/{model}/{subset}_score-distribution.png
[ ] plot dot chart (length, score). highlight score of gold step.
    -> save as one dot chart at ablation/{model}/{subset}_length-score-chart.png
[ ] plot distance distribution (gold vs. predicted steps)
    -> save as one dot chart at ablation/{model}/{subset}_distance-chart.png   
[ ] compute agent prediction with aggregated score from trajectory.
"""

## task 1

In [ ]:
import json
import math
import re
from pathlib import Path

import numpy as np
import pandas as pd

# ── config ────────────────────────────────────────────────────────────────────
# RESULTS_DIR = Path("ablation/qwen3-8b/hand-crafted")
RESULTS_DIR = Path("ablation/llama-3.1-8b/algorithm-generated")
NORM_TYPE   = "l1_norm"
K           = 3
N_LAYERS    = 36

layer_configs = {f"layer/{i}": rf"model\.layers\.{i}\."           for i in range(N_LAYERS)}
mlp_configs   = {f"mlp/{i}":   rf"model\.layers\.{i}\.mlp\."     for i in range(N_LAYERS)}
attn_configs  = {f"attn/{i}":  rf"model\.layers\.{i}\.self_attn\." for i in range(N_LAYERS)}
all_configs   = layer_configs | mlp_configs | attn_configs | {"lm_head": "lm_head"}

config_names   = list(all_configs.keys())
config_patterns = list(all_configs.values())

# ── pre-load ──────────────────────────────────────────────────────────────────
trajectories = [json.loads(f.read_text()) for f in sorted(RESULTS_DIR.glob("*.json"))]

# ── step 1: precompute config→param mask once ─────────────────────────────────
# param names are the same across all logs — grab from first valid log
sample_stats = next(
    log["statistics"]
    for data in trajectories
    for log in data["logs"]
    if log.get("statistics")
)
param_names = list(sample_stats.keys())
n_params    = len(param_names)
n_configs   = len(config_names)

# (n_configs, n_params) boolean matrix
mask_matrix = np.array(
    [[bool(re.search(pat, p)) for p in param_names] for pat in config_patterns],
    dtype=np.float64,
)  # precomputed once — ~109 configs × ~300 params, negligible

n_params_per_config = mask_matrix @ np.array(
    [sample_stats[p]["n_params"] for p in param_names], dtype=np.float64
)  # (n_configs,)

# ── step 2: score all configs for a log in one pass ───────────────────────────
def score_log_all_configs(log: dict) -> np.ndarray:
    """Returns (n_configs,) array of scores for the given log."""
    stats = log["statistics"]
    if NORM_TYPE == "l2_norm":
        vals = np.array([stats[p]["l2_norm_sq"] for p in param_names], dtype=np.float64)
        # (n_configs,): sum l2_sq per config, then sqrt and normalize
        sums = mask_matrix @ vals                          # (n_configs,)
        with np.errstate(invalid="ignore"):
            scores = np.sqrt(sums) / np.where(n_params_per_config > 0, n_params_per_config, np.nan)
    else:
        vals = np.array([stats[p]["l1_norm"] for p in param_names], dtype=np.float64)
        sums = mask_matrix @ vals
        with np.errstate(invalid="ignore"):
            scores = sums / np.where(n_params_per_config > 0, n_params_per_config, np.nan)
    return scores  # (n_configs,)

# ── sweep: single pass over trajectories ─────────────────────────────────────
# accumulators: (n_configs,)
step_correct_sum  = np.zeros(n_configs)
agent_correct_sum = np.zeros(n_configs)
n_total           = 0

for data in trajectories:
    meta          = data["metadata"]
    mistake_step  = int(meta["mistake_step"])
    mistake_agent = meta["mistake_agent"]
    step_roles    = {s["step_idx"]: s["role"] for s in data["steps"]}

    valid_logs = [log for log in data["logs"] if log.get("statistics")]
    if not valid_logs:
        continue

    # (n_logs, n_configs)
    score_matrix = np.stack([score_log_all_configs(log) for log in valid_logs])
    step_indices = np.array([log["step_idx"] for log in valid_logs])  # (n_logs,)

    # For each config, find top-K lowest-score steps
    # argsort along n_logs axis → (n_logs, n_configs)
    ranked = np.argsort(score_matrix, axis=0)[:K]  # (K, n_configs)
    pred_step_matrix = step_indices[ranked]         # (K, n_configs)

    step_correct  = np.any(pred_step_matrix == mistake_step, axis=0).astype(float)

    # agent_correct: check if mistake_agent is among pred_agents for each config
    def agent_correct_for_config(pred_steps_col):
        pred_agents = [step_roles.get(idx, "unknown") for idx in pred_steps_col]
        pred_agents = ["Orchestrator" if "orchestrator" in a.lower() else a for a in pred_agents]
        return float(mistake_agent in pred_agents)

    agent_correct = np.array([
        agent_correct_for_config(pred_step_matrix[:, c]) for c in range(n_configs)
    ])

    step_correct_sum  += step_correct
    agent_correct_sum += agent_correct
    n_total           += 1

# ── results ───────────────────────────────────────────────────────────────────
sweep_df = pd.DataFrame({
    "config":    config_names,
    "group":     [c.split("/")[0] for c in config_names],
    "layer_idx": pd.array([c.split("/")[1] if "/" in c else None for c in config_names]),
    "n":         n_total,
    "step_acc":  step_correct_sum  / n_total,
    "agent_acc": agent_correct_sum / n_total,
}).sort_values("step_acc", ascending=False).reset_index(drop=True)

# print(f"=== Top-10 by step_acc (K={K}, {NORM_TYPE}) ===")
# print(sweep_df.assign(
#     step_acc  = sweep_df["step_acc"].map("{:.1%}".format),
#     agent_acc = sweep_df["agent_acc"].map("{:.1%}".format),
# ).head(10).to_string(index=False))

# print(f"\n=== Best per group ===")
# best = (
#     sweep_df.sort_values("step_acc", ascending=False)
#             .groupby("group", sort=False).first().reset_index()
#             [["group", "config", "layer_idx", "step_acc", "agent_acc"]]
# )
# best["step_acc"]  = best["step_acc"].map("{:.1%}".format)
# best["agent_acc"] = best["agent_acc"].map("{:.1%}".format)
# print(best.to_string(index=False))